In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from langchain import hub
from langchain_core.agents import AgentFinish

prompt_agent = hub.pull('hwchase17/openai-tools-agent')

prompt_agent

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='

In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [4]:
# Step 1: Understand the Problem and the Data
# Step 2: Import and Inspect the Data
# Step 3: Handling Missing Values
# Step 4: Explore Data Characteristics
# Step 5: Performing Data Transformation
# Step 6: Handling Outliers

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zeeshier/student-admission-records")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Aviral Verma\.cache\kagglehub\datasets\zeeshier\student-admission-records\versions\1


In [6]:
path = "C:\\Users\\Aviral Verma\\.cache\\kagglehub\\datasets\\zeeshier\\student-admission-records\\versions\\1\\student_admission_record_dirty.csv"

In [7]:
import subprocess
from langchain_core.tools import tool
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy import stats
import os


class EDAToolpy:

    path: str
    dframe: pd.core.frame.DataFrame     
    
    def __init__(self):
        pass
        
    @staticmethod  
    @tool
    def helper():
        """Help Me With EDA"""
        return "Please Provide your path. (Dont type anything elese) "

    @staticmethod
    @tool 
    def read_from_path(input : str):
        """Reading The CSV"""
        if not os.path.exists(input):
            return None
        EDAToolpy.path = input
        EDAToolpy.dframe = pd.read_csv(path)
        return "Ask Question!"

    @staticmethod
    @tool
    def describe_data_set():
        """Describe Data Set"""
        EDAToolpy.dframe = EDAToolpy.dframe.convert_dtypes()
        print(EDAToolpy.dframe.info())
        print(EDAToolpy.dframe.describe())
        print(EDAToolpy.dframe.nunique())

    @staticmethod
    @tool
    def handling_missing_value():
        """describe the dataset read"""
        print(EDAToolpy.dframe.isnull().sum())
        len_df = len(EDAToolpy.dframe)
        col_cnt = 0
        drop_col = []
        for null_cnt in EDAToolpy.dframe.isnull().sum():
            if null_cnt > 0.16 * len_df:
                drop_col.append(EDAToolpy.dframe.columns[col_cnt])
            col_cnt = col_cnt + 1
        EDAToolpy.dframe.drop(drop_col, axis = 1, inplace=True)
        EDAToolpy.dframe.fillna(method='ffill', inplace=True)

    @staticmethod
    @tool
    def explore_and_visualize_data_characteristics():
        """Exploring The Dataset"""
        sns.heatmap(EDAToolpy.dframe.corr(method='pearson', numeric_only=True, min_periods=1))
        cat_col = []
        num_col = []
        for col in EDAToolpy.dframe.columns:
            if type(EDAToolpy.dframe[col][0]) is str:
                cat_col.append(col)
            else:
                num_col.append(col)
        print(cat_col)
        chi_score = {}
        for col1 in cat_col:
            for col2 in cat_col:
                chi2, p, dof, expected = chi2_contingency((pd.crosstab(EDAToolpy.dframe[col1], EDAToolpy.dframe[col2]).values))
                print(f'Chi-square Statistic for {col1} and {col2} : {chi2} ,p-value: {p}')
    
    
        plt.show()

    @staticmethod
    @tool
    def handling_outliers():
        """Handling The Outliers"""
        cat_col = []
        num_col = []
        for col in EDAToolpy.dframe.columns:
            if type(EDAToolpy.dframe[col][0]) is str:
                cat_col.append(col)
            else:
                num_col.append(col)
        
        for col in num_col:
            EDAToolpy.dframe = EDAToolpy.dframe[(np.abs(stats.zscore(EDAToolpy.dframe[col], nan_policy='omit')) < 3)]
    
    
        print('Categorical Values Dont have specific Outliers Detection')
        
    @staticmethod
    @tool 
    def transforming_data():
        """Transforming The Data"""
        cat_col = []
        num_col = []
        for col in EDAToolpy.dframe.columns:
            if type(EDAToolpy.dframe[col][0]) is str:
                cat_col.append(col)
            else:
                num_col.append(col)
    
        for col in num_col:
            EDAToolpy.dframe[col]=(EDAToolpy.dframe[col]-EDAToolpy.dframe[col].mean())/EDAToolpy.dframe[col].std()
    
        one_hot_col = []
        for col in cat_col:
            # apply unique criteria
            if len(EDAToolpy.dframe[col].unique()) < 20:
                one_hot_col.append(col)
                
        EDAToolpy.dframe = pd.get_dummies(EDAToolpy.dframe, columns=one_hot_col, drop_first=True)
        print(EDAToolpy.dframe.columns)

In [8]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [9]:
llm_with_tool = llm.bind_tools([EDAToolpy.helper,EDAToolpy.read_from_path, EDAToolpy.describe_data_set, EDAToolpy.handling_missing_value, EDAToolpy.explore_and_visualize_data_characteristics, EDAToolpy.handling_outliers, EDAToolpy.transforming_data])

In [10]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
tools = [EDAToolpy.helper,EDAToolpy.read_from_path, EDAToolpy.describe_data_set, EDAToolpy.handling_missing_value, EDAToolpy.explore_and_visualize_data_characteristics, EDAToolpy.handling_outliers, EDAToolpy.transforming_data]

agent = create_tool_calling_agent(llm_with_tool, tools, prompt_agent)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [12]:

while True:
    prompt = input("Hi")
    if prompt == "@EXIT":
        break
    if prompt == "@EDA":
        agent_executor.invoke({"input": "Could you help me with the EDA"})
        continue
    agent_executor.invoke({"input": prompt})

Hi @EDA




> Entering new AgentExecutor chain...

Invoking: `helper` with `{}`


Please Provide your path. (Dont type anything elese) Please provide your path. (Don't type anything else)

> Finished chain.


Hi C:\\Users\\Aviral Verma\\.cache\\kagglehub\\datasets\\zeeshier\\student-admission-records\\versions\\1\\student_admission_record_dirty.csv




> Entering new AgentExecutor chain...

Invoking: `read_from_path` with `{'input': 'C:\\Users\\Aviral Verma\\.cache\\kagglehub\\datasets\\zeeshier\\student-admission-records\\versions\\1\\student_admission_record_dirty.csv'}`


Ask Question!It seems that I cannot access the file directly from your local system. However, you can upload the CSV file here, and I can assist you with it. Please upload the file so we can proceed.

> Finished chain.


KeyboardInterrupt: Interrupted by user